In [1]:
#Browsing indi table in DB for potential appendiceal cancer analyses.

import sqlite3
import pandas as pd
from pathlib import Path

project_root = Path.cwd().parents[0]
db_path = project_root / "database" / "faers.db"
conn = sqlite3.connect(db_path)

indi = pd.read_sql_query("SELECT * FROM indi LIMIT 10000", conn)

In [ ]:
print(indi.shape)
print(indi.dtypes)

primaryid         int64
caseid            int64
indi_drug_seq     int64
indi_pt             str
source_quarter      str
dtype: object

In [ ]:
indi.isnull().sum()  

primaryid         0
caseid            0
indi_drug_seq     0
indi_pt           0
source_quarter    0
dtype: int64

In [5]:
indi.nunique()


primaryid         1322
caseid            1322
indi_drug_seq      592
indi_pt            570
source_quarter       1
dtype: int64

In [6]:
indi['indi_pt'].value_counts().head(20)


indi_pt
Product used for unknown indication         4746
Rheumatoid arthritis                         948
Attention deficit hyperactivity disorder     151
Asthma                                       129
Hypertension                                 123
Pain                                         106
HIV infection                                101
Autism spectrum disorder                      93
Bipolar disorder                              93
Narcolepsy                                    82
Schizophrenia                                 79
Depression                                    77
Prophylaxis                                   72
Oppositional defiant disorder                 64
Cataplexy                                     64
Coronary artery disease                       64
Juvenile idiopathic arthritis                 62
Affective disorder                            60
Parkinson's disease                           53
Sleep disorder                                50
Name: count,

In [12]:
cancer_indi_counts = pd.read_sql_query("""
SELECT indi_pt, COUNT(*) AS reports
FROM indi
WHERE indi_pt LIKE '%appendic%'
   OR indi_pt LIKE '%colorect%'
   OR indi_pt LIKE '%periton%'
   OR indi_pt LIKE '%pseudomyxoma%'
   OR indi_pt LIKE '%cecal%'
   OR indi_pt LIKE '%mucinous%'
   OR indi_pt LIKE '%colon cancer%'
GROUP BY indi_pt
ORDER BY reports DESC
""", conn)

cancer_indi_counts



,indi_pt,reports
0,Colon cancer,6647
1,Colorectal cancer metastatic,5273
2,Colorectal cancer,3094
3,Colon cancer metastatic,1236
4,Malignant peritoneal neoplasm,736
...,...,...
66,Complicated appendicitis,2
67,Borderline mucinous tumour of ovary,2
68,Peritoneal lavage,1
69,Appendicitis noninfective,1


In [14]:
print(cancer_indi_counts.to_string())


                                                   indi_pt  reports
0                                             Colon cancer     6647
1                             Colorectal cancer metastatic     5273
2                                        Colorectal cancer     3094
3                                  Colon cancer metastatic     1236
4                            Malignant peritoneal neoplasm      736
5                                 Metastases to peritoneum      605
6                                      Peritoneal dialysis      575
7                                Colorectal adenocarcinoma      460
8                                              Peritonitis      358
9                                    Colon cancer stage IV      310
10                             Colorectal cancer stage III      234
11                                  Colon cancer stage III      171
12                                   Peritonitis bacterial      138
13                         HER2 positive colorec

In [17]:
cancer_terms = [
    'Colon cancer',
    'Colorectal cancer metastatic',
    'Colorectal cancer',
    'Colon cancer metastatic',
    'Malignant peritoneal neoplasm',
    'Metastases to peritoneum',
    'Colorectal adenocarcinoma',
    'Colon cancer stage IV',
    'Colorectal cancer stage III',
    'Colon cancer stage III',
    'HER2 positive colorectal cancer',
    'Colorectal cancer stage IV',
    'Peritoneal mesothelioma malignant',
    'Colon cancer stage II',
    'Colon cancer recurrent',
    'Colorectal cancer stage II',
    'Colorectal neoplasm',
    'Peritoneal sarcoma',
    'Colorectal cancer recurrent',
    'Peritoneal neoplasm',
    'Mucinous adenocarcinoma of appendix',
    'Peritoneal carcinoma metastatic',
    'Pseudomyxoma peritonei',
    'Peritoneal mesothelioma malignant recurrent',
    'Intraductal papillary mucinous neoplasm',
    'Colorectal cancer stage I',
    'Colon cancer stage I',
    'Appendiceal mucinous neoplasm',
    'Hereditary non-polyposis colorectal cancer syndrome',
]

cancer_drugs = pd.read_sql_query(f"""
SELECT 
    d.drugname,
    COUNT(DISTINCT d.primaryid) AS reports
FROM drug d
JOIN indi i ON d.primaryid = i.primaryid
WHERE i.indi_pt IN ({','.join(['?' for _ in cancer_terms])})
GROUP BY d.drugname
ORDER BY reports DESC
LIMIT 30
""", conn, params=cancer_terms)

cancer_drugs


,drugname,reports
0,OXALIPLATIN,1954
1,FLUOROURACIL,1844
2,BEVACIZUMAB,1015
3,LEUCOVORIN,884
4,CAPECITABINE,875
5,FRUQUINTINIB,673
6,IRINOTECAN,614
7,LONSURF,567
8,VECTIBIX,563
9,CETUXIMAB,466


In [15]:
reaction_counts = pd.read_sql_query("""
SELECT
    i.indi_pt,
    r.pt AS reaction,
    COUNT(DISTINCT r.primaryid) AS reports
FROM indi i
JOIN reac r ON i.primaryid = r.primaryid
WHERE i.indi_pt IN (
    'Colon cancer',
    'Colorectal cancer',
    'Colorectal cancer metastatic',
    'Colon cancer metastatic',
    'Malignant peritoneal neoplasm',
    'Pseudomyxoma peritonei',
    'Mucinous adenocarcinoma of appendix'
)
GROUP BY i.indi_pt, r.pt
ORDER BY i.indi_pt, reports DESC
""", conn)

reaction_counts.head(20)


,indi_pt,reaction,reports
0,Colon cancer,Death,393
1,Colon cancer,Fatigue,316
2,Colon cancer,Diarrhoea,285
3,Colon cancer,Off label use,249
4,Colon cancer,Nausea,224
5,Colon cancer,Colon cancer,173
6,Colon cancer,Decreased appetite,171
7,Colon cancer,Asthenia,153
8,Colon cancer,Blood pressure increased,125
9,Colon cancer,Vomiting,123


In [16]:
ranked_reactions = pd.read_sql_query("""
WITH reaction_counts AS (
    SELECT
        i.indi_pt,
        r.pt AS reaction,
        COUNT(DISTINCT r.primaryid) AS reports
    FROM indi i
    JOIN reac r ON i.primaryid = r.primaryid
    WHERE i.indi_pt IN (
        'Colon cancer',
        'Colorectal cancer',
        'Colorectal cancer metastatic',
        'Colon cancer metastatic',
        'Malignant peritoneal neoplasm',
        'Pseudomyxoma peritonei',
        'Mucinous adenocarcinoma of appendix'
    )
    GROUP BY i.indi_pt, r.pt
),
ranked AS (
    SELECT *,
        RANK() OVER (
            PARTITION BY indi_pt
            ORDER BY reports DESC
        ) AS reaction_rank
    FROM reaction_counts
)
SELECT * FROM ranked
WHERE reaction_rank <= 5
ORDER BY indi_pt, reaction_rank
""", conn)

ranked_reactions


,indi_pt,reaction,reports,reaction_rank
0,Colon cancer,Death,393,1
1,Colon cancer,Fatigue,316,2
2,Colon cancer,Diarrhoea,285,3
3,Colon cancer,Off label use,249,4
4,Colon cancer,Nausea,224,5
5,Colon cancer metastatic,Off label use,48,1
6,Colon cancer metastatic,Fatigue,31,2
7,Colon cancer metastatic,Diarrhoea,29,3
8,Colon cancer metastatic,Neoplasm progression,29,3
9,Colon cancer metastatic,Asthenia,28,5


In [18]:
drug_name_map = {
    'AVASTIN': 'BEVACIZUMAB',
    'ERBITUX': 'CETUXIMAB',
    'VECTIBIX': 'PANITUMUMAB',
    'LEUCOVORIN CALCIUM': 'LEUCOVORIN',
    'LEVOLEUCOVORIN CALCIUM': 'LEUCOVORIN',
    'IRINOTECAN HYDROCHLORIDE': 'IRINOTECAN',
    'LONSURF': 'TRIFLURIDINE/TIPIRACIL',
    'BRAFTOVI': 'ENCORAFENIB',
    'MEKTOVI': 'BINIMETINIB',
    'STIVARGA': 'REGORAFENIB',
    'ZEJULA': 'NIRAPARIB',
    'KEYTRUDA': 'PEMBROLIZUMAB',
}

cancer_drugs['drugname_normalized'] = cancer_drugs['drugname'].replace(drug_name_map)

cancer_drugs_normalized = (
    cancer_drugs
    .groupby('drugname_normalized')['reports']
    .sum()
    .reset_index()
    .sort_values('reports', ascending=False)
)

cancer_drugs_normalized


,drugname_normalized,reports
18,OXALIPLATIN,1954
11,FLUOROURACIL,1844
14,LEUCOVORIN,1486
4,BEVACIZUMAB,1366
13,IRINOTECAN,996
6,CAPECITABINE,875
7,CETUXIMAB,807
19,PANITUMUMAB,757
12,FRUQUINTINIB,673
23,TRIFLURIDINE/TIPIRACIL,567


In [19]:
drug_reactions_in_cancer = pd.read_sql_query("""
WITH cancer_reports AS (
    SELECT DISTINCT primaryid
    FROM indi
    WHERE indi_pt IN (
        'Colon cancer',
        'Colorectal cancer metastatic',
        'Colorectal cancer',
        'Colon cancer metastatic',
        'Malignant peritoneal neoplasm',
        'Pseudomyxoma peritonei',
        'Mucinous adenocarcinoma of appendix'
    )
),
drug_reaction_counts AS (
    SELECT
        d.drugname,
        r.pt AS reaction,
        COUNT(DISTINCT d.primaryid) AS reports
    FROM drug d
    JOIN reac r ON d.primaryid = r.primaryid
    JOIN cancer_reports cr ON d.primaryid = cr.primaryid
    WHERE d.drugname IN (
        'OXALIPLATIN', 'FLUOROURACIL', 'BEVACIZUMAB', 'AVASTIN',
        'LEUCOVORIN', 'LEUCOVORIN CALCIUM', 'IRINOTECAN',
        'IRINOTECAN HYDROCHLORIDE', 'CAPECITABINE'
    )
    GROUP BY d.drugname, r.pt
),
ranked AS (
    SELECT *,
        RANK() OVER (
            PARTITION BY drugname
            ORDER BY reports DESC
        ) AS reaction_rank
    FROM drug_reaction_counts
)
SELECT * FROM ranked
WHERE reaction_rank <= 5
ORDER BY drugname, reaction_rank
""", conn)

drug_reactions_in_cancer


,drugname,reaction,reports,reaction_rank
0,AVASTIN,Diarrhoea,50,1
1,AVASTIN,Death,44,2
2,AVASTIN,Fatigue,43,3
3,AVASTIN,Disease progression,30,4
4,AVASTIN,Nausea,25,5
5,BEVACIZUMAB,Diarrhoea,89,1
6,BEVACIZUMAB,Neutropenia,87,2
7,BEVACIZUMAB,Colorectal cancer metastatic,84,3
8,BEVACIZUMAB,Nausea,76,4
9,BEVACIZUMAB,Anaemia,65,5


In [20]:
drug_outcomes_in_cancer = pd.read_sql_query("""
WITH cancer_reports AS (
    SELECT DISTINCT primaryid
    FROM indi
    WHERE indi_pt IN (
        'Colon cancer',
        'Colorectal cancer metastatic',
        'Colorectal cancer',
        'Colon cancer metastatic',
        'Malignant peritoneal neoplasm',
        'Pseudomyxoma peritonei',
        'Mucinous adenocarcinoma of appendix'
    )
),
drug_outcome_counts AS (
    SELECT
        d.drugname,
        o.outc_cod AS outcome,
        COUNT(DISTINCT d.primaryid) AS reports
    FROM drug d
    JOIN outc o ON d.primaryid = o.primaryid
    JOIN cancer_reports cr ON d.primaryid = cr.primaryid
    WHERE d.drugname IN (
        'OXALIPLATIN', 'FLUOROURACIL', 'BEVACIZUMAB', 'AVASTIN',
        'LEUCOVORIN', 'LEUCOVORIN CALCIUM', 'IRINOTECAN',
        'IRINOTECAN HYDROCHLORIDE', 'CAPECITABINE'
    )
    GROUP BY d.drugname, o.outc_cod
),
ranked AS (
    SELECT *,
        RANK() OVER (
            PARTITION BY drugname
            ORDER BY reports DESC
        ) AS outcome_rank
    FROM drug_outcome_counts
)
SELECT * FROM ranked
ORDER BY drugname, outcome_rank
""", conn)

drug_outcomes_in_cancer


,drugname,outcome,reports,outcome_rank
0,AVASTIN,OT,177,1
1,AVASTIN,DE,94,2
2,AVASTIN,HO,88,3
3,AVASTIN,LT,9,4
4,AVASTIN,DS,6,5
5,BEVACIZUMAB,OT,677,1
6,BEVACIZUMAB,HO,336,2
7,BEVACIZUMAB,DE,122,3
8,BEVACIZUMAB,LT,76,4
9,BEVACIZUMAB,DS,9,5
